# Timika Score Prediction

## Overview

This notebook implements an automated deep learning pipeline to estimate the **Timika score**—a clinically established semi-quantitative measure of tuberculosis (TB) disease severity on chest radiographs (CXR), scored on a 0–140 scale. This work is part of a pilot study assessing machine learning (ML) approaches to predict TB treatment outcomes using the IeDEA-Southern Africa (SA) cohort.

## Background / Study Context

Post-treatment morbidity and disengagement from TB care remain major challenges in high HIV-prevalence settings. This pilot study evaluated tabular, image-only, and multimodal ML approaches for predicting treatment continuity and post-treatment radiographic sequelae using data from the **IeDEA-Southern Africa consortium**, a multi-country cohort combining structured clinical/radiological data with linked CXRs from **1,438 adults (≥15 years)** with pulmonary TB enrolled between **October 2022 and September 2024** across five countries: **Malawi, Mozambique, South Africa, Zambia, and Zimbabwe**.

The Timika score pipeline was specifically developed to automate radiographic severity scoring, which is normally assessed manually, and to test whether models trained on public datasets generalize to a real-world, multi-country African cohort.

## Pipeline Architecture

The Timika score is decomposed into three clinically interpretable sub-tasks executed in sequence:

1. **Lung Detection / Segmentation** — a U-Net-based model identifies and delineates lung fields to define total lung area.
2. **Lesion Detection** — an ensemble of nnU-Net and YOLOv8 identifies TB-related abnormalities within the segmented lungs, from which the percentage of affected lung area is calculated.
3. **Cavitation Detection** — a DenseNet121 model classifies the presence/absence of lung cavitation.

The final predicted Timika score combines the estimated **percentage of affected lung area** with a **fixed increment of +40 points for cavitation**, consistent with the clinical scoring definition. Implementation details build on prior work by Kantipudi et al.:
Kantipudi K, et al. Automated Pulmonary Tuberculosis Severity Assessment on Chest X-rays. J Imaging Inform Med 37(5), 2173–2185, 2024.

## Data Used

**Training data** (public datasets, chosen for larger sample size to support model development):
- Shenzhen chest X-ray dataset (NLM)
- Montgomery County (MC) chest X-ray dataset (NLM) — combined with Shenzhen: 802 images
- NIAID TB Portals dataset: 13,763 images

**Independent test/evaluation data:**
- IeDEA-Southern Africa cohort: 1,583 CXRs, used as an external, out-of-domain test set (not seen during training) to assess real-world generalization rather than in-sample performance. This subset was drawn primarily from Malawi, South Africa, and Zambia, as image quality (low-exposure images, digitized film scans) limited usability from other sites.

## Results Summary

- **Mean Absolute Error (MAE): 27.21** (SD 21.86) on the 0–140 Timika scale when evaluated against reference scores on the independent IeDEA-SA test set — indicating **moderate agreement**.
- **Signed error analysis**: mean bias of **−11.36** (SD 33.00), indicating a slight tendency toward **underestimating disease severity**, with both under- and overestimation observed across cases.
- Despite domain shift from differences in projection, exposure, positioning, and image quality across sites, the pipeline demonstrated **modest but clear agreement** with reference scores, supporting the feasibility of radiology-focused deep learning within IeDEA-SA.

## Interpretation Notes

- This is a **pilot/feasibility** result, not a validated clinical tool. Performance was likely affected by domain shift between curated public training data and heterogeneous, routine-care IeDEA-SA images.
- Findings suggest that improved **cross-site metadata capture** (e.g., projection type, equipment type) and potential **local calibration/adaptation** would likely improve future performance.

## Data Access

IeDEA-SA raw tabular and image data require a project concept form submitted to and approved by the IeDEA-SA consortium (per its Principles of Collaboration and IRB requirements): https://www.iedea-sa.org/contact-us/
Public training datasets:
- TB Portals: https://tbportals.niaid.nih.gov
- Shenzhen dataset: NLM LHNCBC data portal (https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Shenzhen-Hospital-CXR-Set/index.html)
- Montgomery County dataset: NLM LHNCBC data portal (https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomerySet/index.html)

## Funding & Acknowledgments

This work was supported by:
- The **Office of AIDS Research (OAR), National Institutes of Health (NIH)** — primary funder of this work.
- The **Lister Hill National Center for Biomedical Communications (LHNCBC), National Library of Medicine (NLM), NIH**.
- Federal funds from the **National Institute of Allergy and Infectious Diseases (NIAID), NIH**, Department of Health and Human Services, under BCBB Support Services Contract HHSN316201300006W/75N93022F00001 to Guidehouse Digital.
- Computation was performed on the **Biowulf Linux cluster** at NIH, Bethesda, MD (http://biowulf.nih.gov).

Contributions of NIH authors are considered Works of the United States Government. Findings and conclusions are those of the authors and do not necessarily reflect the views of the NIH or U.S. Department of Health and Human Services. The authors declare no conflicts of interest.

## Ethics

All participating IeDEA-SA sites obtained approval from local institutional review boards/ethics committees, and all study participants provided written informed consent.

## Citation

Cid V, Bui V, Kantipudi K, Yaniv Z, Jaeger S. Assessing the Potential of Machine Learning for Predicting Tuberculosis Treatment Outcome in a Multi-Country Cohort in Southern Africa. 54th Annual Applied Imagery Pattern Recognition Workshop, Washington, DC. Springer LNCS, 2026.

## Point of Contact:

Vy Bui, Stefan Jaeger
National Library of Medicine
National Institutes of Health
8600 Rockville Pike
Bethesda, MD 20894
Email: {vy.bui, stefan.jaeger}@nih.gov



Runs the three-model TB chest-X-ray pipeline end-to-end and computes the **Timika score** per image:

1. **Lung segmentation** (heart excluded) &rarr; binary lung mask
2. **Lesion segmentation** (nnU-Net + YOLOv8m ensemble) &rarr; binary lesion mask
3. **Cavity classification** &rarr; cavity probability + 0/1 label

Then:

```
PAL    = (lung_mask ∩ lesion_mask).sum() * 100 / lung_mask.sum()   # percent of lung affected
Timika = PAL + 40 * cavity_label                                   # cavity adds 40
```

**Self-contained & portable.** 
- `timika_lib/` -  model code
- `weights/` - all model weights
- `sample.csv` + `data/` - inputs
- run the install cell once (installs deps into `./pylibs`), then run the cells top to bottom.

In [ ]:
# --- Install dependencies (run once) -------------------------------------
import sys, os, subprocess
from pathlib import Path

# Locate the bundle (folder containing requirements.txt), robust to a changed cwd.
_req = next((p / 'requirements.txt' for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'requirements.txt').exists()), None)
assert _req is not None, 'requirements.txt not found — run the notebook from the timika_pipeline/ bundle.'
BUNDLE_DIR = _req.parent

PYLIBS = BUNDLE_DIR / 'pylibs' # installed packages go here (self-contained)
_tmp   = BUNDLE_DIR / '.tmp';   _tmp.mkdir(exist_ok=True)

import hashlib, shutil
_want  = hashlib.sha256(_req.read_bytes()).hexdigest()
_stamp = PYLIBS / '.requirements.sha256'
_have  = _stamp.read_text().strip() if _stamp.is_file() else None

if _have == _want:
    print(f'pylibs/ already matches requirements.txt ({_want[:12]}) - nothing to do.')
else:
    if PYLIBS.is_dir() and any(PYLIBS.iterdir()):
        print(f'requirements.txt changed -> rebuilding {PYLIBS} from scratch '
              '(restart the kernel afterwards if anything was already imported)')
        shutil.rmtree(PYLIBS)
    PYLIBS.mkdir(parents=True, exist_ok=True)
    _env = dict(os.environ, TMPDIR=str(_tmp), PIP_NO_CACHE_DIR='1')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade',
         '--target', str(PYLIBS), '-r', str(_req)],
        env=_env,
    )
    _stamp.write_text(_want)
    print('Installed into:', PYLIBS)

if str(PYLIBS) not in sys.path:
    sys.path.insert(0, str(PYLIBS))
print('If imports below still fail, restart the kernel, then run this cell again '
      '(the pip step will be a fast no-op) followed by Cell 0.')

## Setup: paths, working dirs, imports, device

In [ ]:
import os, sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch

# --- Bundle root = the folder that contains timika_lib/
BUNDLE = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
               if (p / 'timika_lib').is_dir()), None)
assert BUNDLE is not None, (
    'Could not locate the bundle. Run this notebook from the timika_pipeline/ directory '
    '(the one containing timika_lib/).'
)
sys.path.insert(0, str(BUNDLE / 'timika_lib'))
# Make pip-installed deps (from the install cell, into ./pylibs) importable after a restart.
_pylibs = BUNDLE / 'pylibs'
if _pylibs.is_dir() and str(_pylibs) not in sys.path:
    sys.path.insert(0, str(_pylibs))

# --- Block the TensorFlow probe -----------------------------------------------------
# monai's __init__ eagerly imports every submodule; monai.visualize.img2tensorboard
# probes torch.utils.tensorboard, which drags in the system-site-packages tensorflow
# 2.16.2 (built against NumPy 1.x) under our bundled NumPy 2.x -> a wall of _ARRAY_API
# and cuDNN-factory noise. monai catches it (optional_import) and continues, so the
# only cost is the noise. Nothing here uses TF/tensorboard; block the probe outright.
for _blocked in ('tensorflow', 'tensorboard', 'torch.utils.tensorboard'):
    sys.modules.setdefault(_blocked, None)

if not getattr(torch.load, '_weights_only_shim', False):
    _orig_torch_load = torch.load
    def _torch_load(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return _orig_torch_load(*args, **kwargs)
    _torch_load._weights_only_shim = True
    torch.load = _torch_load

# --- Custom YOLOv8 override (adds `prob_masks` to Results) ---------------------------
# The lesion ensemble reads results[0].prob_masks (soft probability masks), which
# ultralytics 8.2.2 does not provide. The original pipeline shipped a patched ultralytics;
# here we overlay the vendored override files (timika_lib/yolov8_override/) onto the
# installed ultralytics:
#   results.py -> ultralytics/engine/results.py
#   ops.py     -> ultralytics/utils/ops.py
#   predict.py -> ultralytics/models/yolo/segment/predict.py
import shutil, importlib.util
_ov_dir = BUNDLE / 'timika_lib' / 'yolov8_override'
_u_spec = importlib.util.find_spec('ultralytics')
if _u_spec is not None and _u_spec.submodule_search_locations:
    _u_root = Path(list(_u_spec.submodule_search_locations)[0])
    _ov_map = {
        'results.py': _u_root / 'engine' / 'results.py',
        'ops.py':     _u_root / 'utils' / 'ops.py',
        'predict.py': _u_root / 'models' / 'yolo' / 'segment' / 'predict.py',
    }
    _ov_changed = False
    for _src_name, _dst in _ov_map.items():
        _src = _ov_dir / _src_name
        if _dst.exists() and _src.read_bytes() != _dst.read_bytes():
            shutil.copyfile(_src, _dst)
            _ov_changed = True
    if _ov_changed and 'ultralytics' in sys.modules:
        print('NOTE: applied YOLOv8 prob_masks override -> RESTART the kernel and re-run '
              'Cell 0 so the patched ultralytics is loaded.')

# --- Working directories ---
DATA      = BUNDLE / 'data' # input images live here
RESULTS   = BUNDLE / 'results'
LUNG_DIR  = RESULTS / 'lung_seg'
LESION_DIR= RESULTS / 'lesion_seg'
WORKDIR   = BUNDLE / 'run' # notebook cwd; makes '../data' -> DATA, '../output' -> OUTPUT_TMP
OUTPUT_TMP= BUNDLE / 'output' # CavityNotCavity writes temp crops to '../output/'
for d in (DATA, LUNG_DIR, LESION_DIR, WORKDIR, OUTPUT_TMP):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

# --- Device selection, with a real kernel-launch smoke test --------------------------
# torch.cuda.is_available() only proves the driver/runtime loaded; it does NOT prove
# this wheel was compiled for this card's arch. A CUDA 13 wheel on a V100 (sm_70) or
# P100 (sm_60) passes is_available() and then dies at the first launch with
# "no kernel image is available" -- which MONAI rewrites as a bogus OOM message six
# cells later. Catch it here, where the diagnosis is obvious.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    try:
        torch.zeros(1, device='cuda').add_(1).cpu()   # allocate, launch, synchronize
    except Exception as _e:
        _cap = torch.cuda.get_device_capability(0)
        print(f'!! GPU present but unusable ({type(_e).__name__}: {_e})')
        print(f'   torch {torch.__version__} built for CUDA {torch.version.cuda}')
        print(f'   this GPU  : {torch.cuda.get_device_name(0)} sm_{_cap[0]}{_cap[1]}')
        print(f'   wheel has : {torch.cuda.get_arch_list()}')
        print('   -> falling back to CPU. To use the GPU, reinstall torch from an index '
              'whose arch list includes this card (see requirements.txt).')
        device = torch.device('cpu')
print('Bundle      :', BUNDLE)
print('Working dir :', Path.cwd())
print('Device      :', device)

# --- Import the model helpers (heavy: torch / monai / ultralytics / nnunetv2) ---
try:
    from segment_lung_cxr.inference.inference_lung_segment import _load_model, _predict_mask
    from segment_tb_cxr.auxiliary.ensemble_nnunet_yolov8m import (
        gen_yolov8_prob_map, gen_nnunet_prob_map, gen_ensembled_yolov8_nnunet_segmentation,
    )
    from cavity_no_cavity.inference.CavityNotCavity import CavityNotCavity
    from ultralytics import YOLO
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
except ImportError as e:
    raise ImportError(
        f'{e}\n\nMissing dependency. Run the install cell at the top of the notebook '
        '(installs into ./pylibs), then restart the kernel and re-run from here.'
    )

def _load_seg_model(model_path, device):
    model = torch.jit.load(str(model_path), map_location=device)
    model.eval()
    return model

print('Imports OK.')

## Weights & constants

All model weights are loaded from the bundled `weights/` folder (fully portable — no machine-specific paths).

In [ ]:
WEIGHTS = BUNDLE / 'weights'   # all model weights are bundled here

def weight(rel):
    p = WEIGHTS / rel
    assert p.exists(), f'Missing weight: {p}. Make sure the weights/ folder is present in the bundle.'
    return p

LUNG_W     = weight('lung_segment.pt')
CAV_LUNG_W = weight('cavity_lung_segment.pt')
YOLO_W     = weight('yolov8.pt')
NNUNET_W   = weight('nnunet/fold_0/nnunet.pth')   # model root = its parent.parent (has dataset.json + plans.json + fold_0/)
CAV_W      = weight('cavity_no_cavity.pt')

CAVITY_THRESHOLD = 0.2   # Youden threshold for the cavity classifier
LUNG_INPUT_SIZE  = (224, 224)
LESION_BIN_THRESHOLD = 0.5

for name, p in [('lung', LUNG_W), ('cavity-lung', CAV_LUNG_W), ('yolov8', YOLO_W), ('nnunet', NNUNET_W), ('cavity', CAV_W)]:
    print(f'{name:12s}: {p}')

## Input

Input filenames are read from **`sample.csv`** in the notebook folder (column `filename`). Put the corresponding image files in `data/`. Edit `sample.csv` to run on your own dataset.

In [ ]:
# Read input filenames from sample.csv (in the same folder as the notebook).
# Accepts a 'filename' or 'input file name' column
CSV_PATH = BUNDLE / 'sample.csv'
assert CSV_PATH.exists(), f'sample.csv not found at {CSV_PATH}'
_csv = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
_col = next((c for c in ('filename', 'input file name') if c in _csv.columns), _csv.columns[0])
image_names = [str(n).strip() for n in _csv[_col].dropna().tolist()]

# Images are referenced as ../data/<name> (relative to the run/ working dir) so every
# model resolves them consistently. Place the listed image files in data/.
images = [f'../data/{n}' for n in image_names]
for p in images:
    assert Path(p).exists(), f'Missing input image: {Path(p).resolve()} (put it in data/)'
print(f'Loaded {len(images)} image(s) from {CSV_PATH.name}:', images)

## Lung segmentation → binary lung mask

In [ ]:
lung_model, _ = _load_model(LUNG_W)
lung_mask_paths = {}
for img in images:
    name = Path(img).stem
    mask = _predict_mask(img, lung_model, device, LUNG_INPUT_SIZE, 0.5)
    out = LUNG_DIR / f'{name}_lung_pred_seg.nii.gz'
    sitk.WriteImage(sitk.Cast(mask, sitk.sitkUInt8), str(out))
    lung_mask_paths[img] = out
    print('lung mask ->', out)

## Lesion segmentation (nnU-Net + YOLOv8m ensemble) → binary lesion mask

In [ ]:
yolo = YOLO(str(YOLO_W), verbose=False)
yolo.to(str(device))

predictor = nnUNetPredictor(device=device, allow_tqdm=False)
predictor.initialize_from_trained_model_folder(
    str(Path(NNUNET_W).parent.parent),          # model root: dir with dataset.json + plans.json + fold_0/
    checkpoint_name=Path(NNUNET_W).name,
    use_folds=(0,),
)

lesion_mask_paths = {}
for img in images:
    name = Path(img).stem
    yolo_prob   = gen_yolov8_prob_map(img, yolo)
    nnunet_prob = gen_nnunet_prob_map(img, predictor)
    gen_ensembled_yolov8_nnunet_segmentation(img, yolo_prob, nnunet_prob, str(LESION_DIR), LESION_BIN_THRESHOLD)
    lesion_mask_paths[img] = LESION_DIR / f'{name}_ensemble_seg.nii.gz'
    print('lesion mask ->', lesion_mask_paths[img])

## Cavity classification

In [ ]:
cav_lung_model = _load_seg_model(CAV_LUNG_W, device)
cav_model      = _load_seg_model(CAV_W, device)
cavity_predictor = CavityNotCavity(
    lung_seg_model=cav_lung_model,
    device=device,
    cavity_not_cavity_models=[cav_model],
    cavity_not_cavity_threshold=CAVITY_THRESHOLD,
)

cavity_rows = {}
for img in images:
    prob, label = cavity_predictor.predict_cavity(img)   # writes a temp crop to ../output/
    cavity_rows[img] = (float(prob), int(label))
    print(f'cavity -> {img}: score={float(prob):.4f} label={int(label)}')

## Timika score

`PAL = (lung ∩ lesion) / lung * 100`, then `Timika = PAL + 40 * cavity_label`.

In [ ]:
def percent_lung_affected(lung_path, lesion_path):
    lung   = sitk.ReadImage(str(lung_path))
    lesion = sitk.ReadImage(str(lesion_path))
    lung_arr   = (sitk.GetArrayFromImage(lung)   > 0).astype(np.uint8)
    lesion_arr = (sitk.GetArrayFromImage(lesion) > 0).astype(np.uint8)
    if lung_arr.shape != lesion_arr.shape:
        # Reconcile grids (the ensemble mask is saved with default geometry): resample
        # the lesion mask onto the lung grid with nearest-neighbor, then compare arrays.
        lesion_r = sitk.Resample(lesion, lung, sitk.Transform(),
                                 sitk.sitkNearestNeighbor, 0, lesion.GetPixelID())
        lesion_arr = (sitk.GetArrayFromImage(lesion_r) > 0).astype(np.uint8)
    denom = int(lung_arr.sum())
    inter = int((lung_arr & lesion_arr).sum())
    return (inter * 100.0 / denom) if denom else 0.0

rows = []
for img in images:
    pal = percent_lung_affected(lung_mask_paths[img], lesion_mask_paths[img])
    prob, label = cavity_rows[img]
    rows.append({
        'input file name': img,
        'new_pal': pal,
        'predicted_cavity_score': prob,
        'cavity_label': label,
        'predicted_timika': pal + 40 * label,
    })

df = pd.DataFrame(rows)
out_csv = RESULTS / f'timika_scores_{datetime.now():%Y%m%d}.csv'
df.to_csv(out_csv, index=False)
print('Saved:', out_csv)
df

## Overlay visualization

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

try:
    import cv2
except Exception:
    cv2 = None

def _load2d(path):
    """Read an image as a 2-D grayscale float array (handles RGB/RGBA/singleton dims)."""
    a = np.squeeze(sitk.GetArrayFromImage(sitk.ReadImage(str(path)))).astype(float)
    if a.ndim == 3:                      # RGB(A) -> luminance
        a = a[..., :3].mean(axis=-1)
    return a

def _mask_like(path, ref):
    """Read a mask as 2-D uint8, resized (nearest) to match `ref` if needed."""
    a = np.squeeze(sitk.GetArrayFromImage(sitk.ReadImage(str(path)))).astype(np.uint8)
    if a.shape != ref.shape:
        if cv2 is not None:
            a = cv2.resize(a, (ref.shape[1], ref.shape[0]), interpolation=cv2.INTER_NEAREST)
        else:                            # fallback nearest-neighbor resize without cv2
            ys = (np.linspace(0, a.shape[0] - 1, ref.shape[0])).round().astype(int)
            xs = (np.linspace(0, a.shape[1] - 1, ref.shape[1])).round().astype(int)
            a = a[np.ix_(ys, xs)]
    return (a > 0).astype(np.uint8)

for img in images:
    orig   = _load2d(img)
    lung   = _mask_like(lung_mask_paths[img], orig)
    lesion = _mask_like(lesion_mask_paths[img], orig)

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(orig, cmap='gray');  ax[0].set_title('original')
    ax[1].imshow(orig, cmap='gray');  ax[1].contour(lung, colors='c', linewidths=0.6); ax[1].set_title('lung mask')
    ax[2].imshow(orig, cmap='gray');  ax[2].contour(lesion, colors='r', linewidths=0.6); ax[2].set_title('lesion mask')

    prob, label = cavity_rows[img]
    pal = float(df.loc[df['input file name'] == img, 'new_pal'].iloc[0])
    fig.suptitle(f'{Path(img).name}   PAL={pal:.1f}%   cavity={label} ({prob:.2f})   Timika={pal + 40*label:.1f}')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()